In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '0' 
import random
random.seed(12)
import numpy as np
np.random.seed(12)
import torch
torch.manual_seed(12)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

import pandas as pd
import sys, importlib
# Robust import for xGATE package (try 'xGATE' then 'xgate')
for _pkg in ("xGATE", "xgate"):
    try:
        utilities = importlib.import_module(f"{_pkg}.utilities")
        for _name in dir(utilities):
            if not _name.startswith("_"):
                globals()[_name] = getattr(utilities, _name)
        break
    except Exception:
        continue
else:
    raise ImportError("Could not import xGATE.utilities. Install the xGATE Python package (see README) or ensure it's on PYTHONPATH.")

# Define the range of i and j
clusters = [1,2]  # i ranges from 1 to 2


test_pathways = ["Apoptosis", "Insulin signaling pathway", "AMPK signaling pathway", "Autophagy", "Cytokine-cytokine receptor interaction", 
                "FoxO signaling pathway", "HIF-1 signaling pathway", "Longevity regulating pathway", "MAPK signaling pathway",
                "Mineral absorption", "PPAR signaling pathway", "Protein processing in endoplasmic reticulum", "Rap1 signaling pathway",
                "cGMP-PKG signaling pathway", "mTOR signaling pathway"]



print("Pancreas AAB clusters")
        
for i in clusters:
    # Print the current cluster 
    print(f'Processing dataset for AAB and cluster{i}of2...')


    # Read the file containing the cell names
    cell_names = pd.read_csv(f'/work/of21_work/pancreas_aab_cluster{i}_final', header=None)
    cell_names = cell_names[0].tolist()  # Convert to a list
    print(len(cell_names))

    # Load your count_matrix DataFrame
    count_matrix = pd.read_csv(f'/work/of21_work/pancreas_aab_final.csv', index_col=0)
    
    # Subset the count_matrix to include only the columns found in cell_names
    subset_count_filtered = count_matrix[cell_names]

    row_means = np.mean(subset_count_filtered, axis=1)

    # Check if any row means are 0 or 1
    rows_with_mean_zero_or_one = (row_means == 0) | (row_means == 1)

    # If you need the indices of such rows
    indices = np.where(rows_with_mean_zero_or_one)[0]

    row_means = np.mean(subset_count_filtered, axis=1)

    # Identify the indices of rows with mean 0 or 1
    rows_to_remove = (row_means == 0) | (row_means == 1)

    # comment out these rows if csv is already stored
    '''
    subset_count_filtered = subset_count_filtered[~rows_to_remove]

    df = pd.DataFrame(subset_count_filtered)
    so = create_sifinet_object(df, rowfeature= True)
    so = quantile_thres2(so)
    so = cal_coexp(so, X = so.data_thres['dt'], X_full = so.data_thres['dt'])
    so = create_network(so, alpha=0.05, manual=False, least_edge_prop=0.01)
    so = filter_lowexp(so, t1=10, t2=0.9, t3=0.9)

    adj_matrix = so.coexp


    sif_ob_test = so

    # Perform the element-wise comparison and assignment
    adj_matrix = pd.DataFrame(np.where(
        np.abs(sif_ob_test.coexp - sif_ob_test.est_ms['mean']) > sif_ob_test.thres,
        np.abs(sif_ob_test.coexp),
        0
    ))

    adj_matrix.index = df.index

    adj_matrix.columns = df.index

    adj_matrix = convert_gene_ids(adj_matrix, "ensembl")

    adj_matrix.to_csv(f'/work/of21_work/adj_matrix_pancreas_aab_cluster{i}_final.csv', index=True)
    '''
    adj_matrix = pd.read_csv(f'/work/of21_work/adj_matrix_pancreas_aab_cluster{i}_final.csv', index_col=0)

    G_s = utilities.create_network_from_adj_matrix(adj_matrix)

    categorized_pathways = utilities.get_categorized_pathways()

    pathway_results = utilities.analyze_pathways(G_s, test_pathways, categorized_pathways ,num_walks=200, max_walk_length = 200)

    pathway_results.to_csv(f'/hpc/group/xielab/of21/re_pathway_results_aab_cluster{i}.csv', index=False)
    #pathway_results.to_csv(f'/work/of21_work/pathway_results_aab_cluster{i}.csv', index=False)


Pancreas AAB clusters
Processing dataset for AAB and cluster1of2...
3066
Pathway: Apoptosis
p-value: 0.04477611940298507
Z-Score: 1.4094116073376328

Pathway: Insulin signaling pathway
p-value: 0.004975124378109453
Z-Score: 48.705614049178266

Pathway: AMPK signaling pathway
p-value: 0.004975124378109453
Z-Score: 34.383138794023665

Pathway: Autophagy
p-value: 0.3482587064676617
Z-Score: -0.1741638056869661

Pathway: Cytokine-cytokine receptor interaction
p-value: 0.27860696517412936
Z-Score: -0.10237688061898667

Pathway: FoxO signaling pathway
p-value: 0.004975124378109453
Z-Score: 19.73006128965189

Pathway: HIF-1 signaling pathway
p-value: 0.004975124378109453
Z-Score: 105.03959907538811

Pathway: Longevity regulating pathway
p-value: 0.004975124378109453
Z-Score: 21.41923094681455

Pathway: MAPK signaling pathway
p-value: 0.004975124378109453
Z-Score: 965.0815318907765

Pathway: Mineral absorption
p-value: 0.004975124378109453
Z-Score: 24.293100769618825

Pathway: PPAR signaling p

In [2]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Load the CSV files for each cluster
df_cluster1 = pd.read_csv('/hpc/group/xielab/of21/re_pathway_results_aab_cluster1.csv')
df_cluster2 = pd.read_csv('/hpc/group/xielab/of21/re_pathway_results_aab_cluster2.csv')

# If your CSVs don't already include a 'Cluster' column, add them:
if 'Cluster' not in df_cluster1.columns:
    df_cluster1['Cluster'] = 'Cluster 1'
if 'Cluster' not in df_cluster2.columns:
    df_cluster2['Cluster'] = 'Cluster 2'

# Combine the two DataFrames into one
df = pd.concat([df_cluster1, df_cluster2], ignore_index=True)

# Apply the Benjamini-Hochberg FDR correction to the "p-value" column
_, adj_pvals, _, _ = multipletests(df["p-value"], alpha=0.05, method='fdr_bh')
df["adjusted p-value"] = adj_pvals

print(df)


                                        pathway   p-value      z-score  \
0                                     Apoptosis  0.044776     1.409412   
1                     Insulin signaling pathway  0.004975    48.705614   
2                        AMPK signaling pathway  0.004975    34.383139   
3                                     Autophagy  0.348259    -0.174164   
4        Cytokine-cytokine receptor interaction  0.278607    -0.102377   
5                        FoxO signaling pathway  0.004975    19.730061   
6                       HIF-1 signaling pathway  0.004975   105.039599   
7                  Longevity regulating pathway  0.004975    21.419231   
8                        MAPK signaling pathway  0.004975   965.081532   
9                            Mineral absorption  0.004975    24.293101   
10                       PPAR signaling pathway  0.970149    -0.272574   
11  Protein processing in endoplasmic reticulum  0.004975   170.331386   
12                       Rap1 signalin

In [3]:
df

,pathway,p-value,z-score,Cluster,adjusted p-value
0,Apoptosis,0.044776,1.409412,Cluster 1,0.055970
1,Insulin signaling pathway,0.004975,48.705614,Cluster 1,0.009328
2,AMPK signaling pathway,0.004975,34.383139,Cluster 1,0.009328
3,Autophagy,0.348259,-0.174164,Cluster 1,0.360268
4,Cytokine-cytokine receptor interaction,0.278607,-0.102377,Cluster 1,0.298507
5,FoxO signaling pathway,0.004975,19.730061,Cluster 1,0.009328
6,HIF-1 signaling pathway,0.004975,105.039599,Cluster 1,0.009328
7,Longevity regulating pathway,0.004975,21.419231,Cluster 1,0.009328
8,MAPK signaling pathway,0.004975,965.081532,Cluster 1,0.009328
9,Mineral absorption,0.004975,24.293101,Cluster 1,0.009328
